# This notebook is part of the LinkedIn Learning Course: [Generative AI: Introduction to Diffusion Models for Text Generation](https://www.linkedin.com/learning/generative-ai-introduction-to-diffusion-models-for-text-generation/building-a-basic-text-diffusion-model-29722011?resume=false&u=35754684)

It is part of the `Section 3: Diffusion Models for Test Generation In Action`.





In [ ]:
#Checking the Python version - for this colab notebook, I use 3.12.11
!python --version

Python 3.12.11


## Loading Modules

Some of the key components used

* `AutoTokenizer`
  * AutoTokenizer is a component within the Hugging Face transformers library that automatically selects and loads the appropriate tokenizer for a given pre-trained model. It streamlines the crucial first step of a Natural Language Processing (NLP) pipeline, which is to convert raw text into a numerical format that a machine learning model can process.

* `AutoModelForCausalLM` is a class
  * It is designed for causal language modeling, which means it's generating text one token at a time. The output will be a sequence of tokens that the model thinks is likely to come next.

* `DDPMScheduler`
  * The DDPMScheduler is an algorithm used in diffusion models to manage the iterative process of converting random noise into a coherent, high-quality image. In the context of the Hugging Face Diffusers library, it refers to the specific implementation based on the **Denoising Diffusion Probabilistic Models (DDPM)** paper.


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from diffusers import DDPMScheduler
import torch
from torch import nn
from typing import Tuple

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # Loading the tokenizer models - bert
text_encoder = AutoModelForMaskedLM.from_pretrained("bert-base-uncased") #
text_encoder.eval()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [ ]:
# Initialize a DDPMScheduler to simulate forward diffusion
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

In [ ]:
def text_to_embedding(text: str) -> torch.Tensor:
  """ Convert text to an embedding.
     The output tensor from this function represents the contextual token
     embeddings for the input text. It is a numerical vector representation that captures
     the semantic meaning and context of each token in the input string.

  Args:
    text (str): The input text to be converted into an embedding.

  Returns:
    (torch.Tensor): the last element of this tuple, which is a single torch.Tensor

  """
  #Tokenize the text
  #  'pt' says return tensor as pytorch tensor
  #  padding means pad the text if the provide text is does not match the max length of token series or truncate if it is longer
  tokens = tokenizer(text, return_tensors='pt', padding=True, truncation=True)

  # torch.no_grad() used to prevent backward calculation -- a computational savings as not using Tensor.backward()
  with torch.no_grad():
    outputs = text_encoder(**tokens, output_hidden_states=True)
  return outputs.hidden_states[-1]

In [ ]:
def add_noise(embedding: torch.Tensor, timestep: int) -> Tuple[torch.Tensor, torch.Tensor]:
  """Adds Gaussian noise to an embedding at a given timestep.

    Args:
        embedding (torch.Tensor): The input tensor (e.g., text or image embedding).
        timestep (int): The timestep for the forward diffusion process.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: A tuple containing the noisy embedding
        and the randomly generated noise tensor.
    """
  noise = torch.rand_like(embedding)
  noisy_embed = noise_scheduler.add_noise(embedding, noise, timestep)
  return noisy_embed,noise

---

## Providing some guidance on `def show_noise_effect()`


### torch.no_grad()

used to prevent backward calculation -- a computational savings as not using Tensor.backward()

### logits = text_encoder(inputs_embeds=noisy_embed).logits

The .logits part of the command accesses the raw, unnormalized output scores from the model's final layer. These logits are not yet converted into probabilities but are real numbers that represent the model's confidence for each possible token in its vocabulary for every position in the sequence.


### logits.argmax(dim=-1)[0]

  * Convert a list of lists of token ids into a list of strings by calling decode `logits.argmax(dim=-1)[0]` extracts the predicted class label (the index of the class with the highest logit) for the first sample within a batch of predictions.

### tokenizer.batch_decode() method,

  * It is commonly found in libraries like Hugging Face Transformers, is used to convert a batch of numerical token IDs back into human-readable text.

Here's a breakdown:

* **Tokenization**:

  * In Natural Language Processing (NLP), text is first converted into numerical representations called "tokens" or "token IDs" so that machine learning models can process it. This process is called tokenization.



* **Decoding:**

    * After a model processes these numerical tokens (e.g., generating a response), the batch_decode() method reverses this process. It takes a list of lists of token IDs (where each inner list represents a sequence of tokens from a different input) and converts each sequence back into its corresponding string of text.

* **Batch Processing:**

  * The "batch" in batch_decode() indicates that it's designed to efficiently decode multiple sequences of token IDs simultaneously, which is common when working with models that process data in batches.


Options: This method often includes options to control the decoding process, such as skip_special_tokens=True to prevent special tokens (like [CLS], [SEP], [PAD]) from appearing in the output text.


In essence, **tokenizer.batch_decode()** is the inverse operation of tokenization, enabling the conversion of numerical model outputs back into understandable language.

---


In [ ]:
def show_noise_effect(embedding: torch.Tensor, timestep: int) -> str:
  """Demonstrates the effect of adding noise to a text embedding by attempting to
    decode the resulting noisy embedding back into human-readable text.
    It is a tool for visualizing the impact of the diffusion process on the embedding space.

    Args:
        embedding (torch.Tensor): The input tensor (e.g., text or image embedding).
        timestep (int): The timestep for the forward diffusion process.

    Returns:
        (str): returns a string of text that is likely to be corrupted or nonsensical due
               to the added noise, depending on the timestep value. At a small timestep,
               the text will be very similar to the original, while at a large timestep,
               it will be completely incoherent, reflecting the principles of a diffusion
               process.
  """
  noisy_embed, _ = add_noise(embedding, timestep)
  with torch.no_grad():
    logits = text_encoder(inputs_embeds=noisy_embed).logits
    noisy_text = tokenizer.batch_decode(logits.argmax(dim=-1)[0], skip_special_tokens=True)[0]
  return noisy_text

### Run some input text

In [ ]:
input_text = "The cat sat on the mat"
clean_embed = text_to_embedding(input_text)
clean_embed

tensor([[[-0.2027,  0.1071, -0.0085,  ..., -0.1198,  0.2286,  0.2522],
         [-0.3376, -0.2804, -0.1846,  ..., -0.3904,  0.9606, -0.6021],
         [-0.3106, -0.0439,  0.2795,  ..., -0.4990,  0.7590,  0.6673],
         ...,
         [-0.5056, -0.2047, -0.1147,  ..., -0.0776,  0.4049, -0.5230],
         [-0.6513, -0.4152, -0.3943,  ...,  0.6858,  0.1113, -0.0406],
         [ 0.7182,  0.1087, -0.3343,  ...,  0.0821, -0.4505, -0.5088]]])

In [ ]:
# Adding increasing levels of noise here to see how the text changes
for timestep in [200,500,800]:
  noisy_text = show_noise_effect(clean_embed, torch.tensor([timestep]))
  print(f"Timestep: {timestep}\nNoisy Text: {noisy_text}\n")

Timestep: 200
Noisy Text: in

Timestep: 500
Noisy Text: and

Timestep: 800
Noisy Text: (



---

## Describing the Denoiser

```denoiser = nn.Sequential(nn.Linear(768,768),nn.ReLU(),nn.Linear(768,768))```

The line of code denoiser = nn.Sequential(nn.Linear(768, 768), nn.ReLU(), nn.Linear(768, 768)) defines a small neural network in PyTorch, storing it within a variable named denoiser.

### What the denoiser does

In the context of a diffusion model, this network is trained to "predict" the noise that was added to an embedding.

* The denoiser receives a noisy_embed as input (a tensor of size 768).

* It processes this noisy data through its two linear layers and **non-linear activation function**.

* The output of the denoiser is a prediction of what the original noise looked like.

* This predicted noise is then used to reverse the diffusion process and generate a cleaner embedding.

### Breaking down the parts:

* nn.Sequential(...):

  * This is a container for modules (layers) that will be added and executed in the order they are passed in the constructor. It allows you to build a network in a simple, readable, and compact way.

* nn.Linear(768, 768):

  * This is the first layer of the network.
    * It is a fully connected layer that performs a linear transformation on the input data.
    * It takes an input tensor of size 768.
    * It outputs a tensor of size 768.
    * In mathematical terms, it calculates `y = xA^T + b`, where x is the input, y is the output, A is the layer's weight matrix, and b is its bias vector.

* nn.ReLU():
  * This is the second layer, acting as an activation function.
  * It introduces a non-linearity into the network, which allows it to learn more complex patterns than a simple linear function.
  * It applies the Rectified Linear Unit function, which is mathematically defined as max(0, x). For every element in the tensor it receives, it returns the value itself if it's positive and returns zero if it's negative.

* nn.Linear(768, 768):
  * This is the third and final layer of the network.
  * It is another fully connected layer that takes the 768-sized output from the nn.ReLU() layer.
  * It performs a second linear transformation.
  * It produces a final output tensor of size 768.

In [ ]:
# Simulate the denoiser
denoiser = nn.Sequential(nn.Linear(768,768),nn.ReLU(),nn.Linear(768,768))

In [ ]:
noisy_embed, true_noise = add_noise(clean_embed, torch.tensor([500]))
print(noisy_embed)
print(true_noise)

tensor([[[ 0.6111,  0.1088,  0.7314,  ...,  0.5074,  0.5668,  0.9461],
         [ 0.6873,  0.5402,  0.7917,  ...,  0.4084,  1.0089,  0.5899],
         [ 0.4878,  0.4155,  0.1561,  ..., -0.0364,  1.0638,  0.4385],
         ...,
         [ 0.4503,  0.7440,  0.6911,  ...,  0.8296,  0.2641,  0.0895],
         [ 0.2014,  0.7222,  0.4111,  ...,  0.1965,  0.5455,  0.5422],
         [ 0.4405,  0.4969,  0.5687,  ...,  0.3524, -0.0138,  0.3419]]])
tensor([[[0.6953, 0.0822, 0.7641,  ..., 0.5631, 0.5238, 0.9120],
         [0.8138, 0.6440, 0.8781,  ..., 0.5387, 0.7716, 0.7892],
         [0.5982, 0.4454, 0.0814,  ..., 0.1071, 0.8873, 0.2628],
         ...,
         [0.6157, 0.8342, 0.7530,  ..., 0.8864, 0.1574, 0.2451],
         [0.3988, 0.8726, 0.5426,  ..., 0.0055, 0.5357, 0.5764],
         [0.2501, 0.4858, 0.6893,  ..., 0.3431, 0.1165, 0.5038]]])


In [ ]:
predicted_noise = denoiser(noisy_embed)
print(predicted_noise)

tensor([[[-0.1358, -0.2668,  0.1466,  ...,  0.1901,  0.0688, -0.1905],
         [-0.1519, -0.2149,  0.2143,  ...,  0.1571,  0.2591, -0.0641],
         [-0.2536, -0.3177,  0.1177,  ...,  0.2733,  0.2401, -0.0582],
         ...,
         [-0.1730, -0.2893,  0.1452,  ...,  0.2828,  0.1224, -0.1282],
         [-0.2136, -0.3033,  0.1345,  ...,  0.2227,  0.1966, -0.0538],
         [-0.0685, -0.2654,  0.1881,  ...,  0.2755,  0.1506, -0.1640]]],
       grad_fn=<ViewBackward0>)


In [ ]:
predicted_clean = noisy_embed - predicted_noise
print(predicted_clean)

tensor([[[ 0.7470,  0.3756,  0.5849,  ...,  0.3172,  0.4980,  1.1366],
         [ 0.8393,  0.7551,  0.5775,  ...,  0.2513,  0.7499,  0.6540],
         [ 0.7414,  0.7332,  0.0384,  ..., -0.3097,  0.8237,  0.4967],
         ...,
         [ 0.6233,  1.0333,  0.5460,  ...,  0.5468,  0.1417,  0.2177],
         [ 0.4149,  1.0255,  0.2766,  ..., -0.0262,  0.3490,  0.5960],
         [ 0.5090,  0.7623,  0.3806,  ...,  0.0770, -0.1644,  0.5059]]],
       grad_fn=<SubBackward0>)


---
## Using the Predicted Noise to Help Recover the original text


This code performs the final step of a text-based diffusion process: decoding a "cleaned" latent representation back into a human-readable text string. It is essentially the inverse of the initial embedding step, where text is converted into a numerical format.

## Here is a breakdown of what each line does:

### `with torch.no_grad():`
This is a context manager in the PyTorch library that temporarily sets all the requires_grad flags to **False** for the model's parameters.

This has two primary benefits:

* **Memory efficiency:** It prevents the tracking of gradients, which saves memory that would otherwise be used for backpropagation.

* **Faster computation:** It speeds up computations by eliminating the need to store and compute gradients, which is essential during the inference (non-training) phase.

This indicates that the model is not being trained but is simply being used to produce an output.

### logits = text_encoder(inputs_embeds=predicted_clean).logits

* The `predicted_clean tensor` is an embedding that has been through the denoising process of the diffusion model. It represents the model's final, denoised prediction of the text embedding.

* This line feeds that `predicted_clean embedding` directly into the text_encoder.

* The `text_encoder` then produces logits, which are the raw, unnormalized output scores for each possible token in the model's vocabulary at each position in the text.


### denoised_text = tokenizer.batch_decode(logits.argmax(dim=-1)[0], skip_special_tokens=True)[0]

* `logits.argmax(dim=-1)`:

  * This finds the index of the highest logit score for each position in the sequence, effectively picking the most likely token ID at every step.

* `[0]`:

  * This selects the first (and in this case, only) sequence from the batch.

* `tokenizer.batch_decode(...)`:

  * This takes the tensor of token IDs and uses the tokenizer's vocabulary to convert them back into a human-readable string.

* `skip_special_tokens=True`:

  * This prevents the inclusion of special tokens (like <PAD> or <EOS>) in the final output string.

* `[0]`:

  * This final index is used to select the first string from the list returned by batch_decode.

### print(denoised_text)
This displays the final, decoded text string to the console.

### In summary,

This code block uses the **text_encoder** and **tokenizer** to convert the `predicted_clean` latent representation from a diffusion model back into a string of text.

The `no_grad()` context indicates this is an **inference** step, not a **training one**.


In [ ]:
with torch.no_grad():
  logits = text_encoder(inputs_embeds=predicted_clean).logits
  denoised_text = tokenizer.batch_decode(logits.argmax(dim=-1)[0], skip_special_tokens=True)[0]
print(denoised_text)


and


In [ ]:
show_noise_effect(clean_embed, torch.tensor([500]))

'"'